# MotherBabyInsight: A Spark-Based E-Commerce Behavior Analytics System

**MotherBabyInsight??? Spark ?????????????**

? Notebook ? `mother_baby_insight.py` ?????????????????????????

## 1. ?????????

???????????????????

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from mother_baby_insight import *

ensure_dirs()
setup_chinese_font()
print(PROJECT_EN_NAME)
print(PROJECT_CN_NAME)

## 2. ???????????

????????????

In [ ]:
print("??????", PROJECT_ROOT)
print("???????", DATA_RAW_DIR)
print("?????", OUTPUT_DIR)
print("?????", CHART_DIR)

## 3. ????????

??????????? Windows ? VMware Linux ????????

In [ ]:
setup_chinese_font()

## 4. SparkSession ??

???? SparkSession???????? Spark??????? Pandas ???

In [ ]:
spark = create_spark_session()

## 5. ????

?? data/raw ??? CSV ???????????????????? 5 ??

In [ ]:
trade_sdf, baby_sdf, trade_df, baby_df = read_raw_data(spark)
display(trade_df.head())
display(baby_df.head())

## 6. ??????

??????????????????????????????????

In [ ]:
save_quality_tables(trade_df, baby_df)
display(pd.read_csv(TABLE_DIR / "data_overview.csv"))
display(pd.read_csv(TABLE_DIR / "missing_values.csv"))

## 7. ????????

????????????IQR ?? buy_mount ????????????????? unknown?

In [ ]:
if spark is not None and trade_sdf is not None and baby_sdf is not None:
    merged_sdf, merged_df = clean_with_spark(spark, trade_sdf, baby_sdf)
else:
    merged_sdf = None
    merged_df = clean_with_pandas(trade_df, baby_df)
save_merged_data(merged_df)
display(merged_df.head())

## 8. ?????????

?? cat1 ???????????????????????????

In [ ]:
CAT1_NAME_MAP

## 9. Spark SQL ?????

?? Spark SQL ? Pandas ?????????????????????

In [ ]:
tables = build_statistics(merged_df, spark, merged_sdf)
display(tables["category_sales_summary"])
display(tables["monthly_sales_summary"].head())
display(tables["weekday_sales_summary"])

## 10. ????

????????? model_dataset.csv?

In [ ]:
model_df = feature_engineering(merged_df)
display(model_df.head())
print("????", PROCESSED_DIR / "model_dataset.csv")

## 11. ?????????? cat1 ??

?? Logistic Regression?Decision Tree?Random Forest?

In [ ]:
cat1_metrics = train_cat1_models(model_df)
display(cat1_metrics)

## 12. ??????????????

?? high_buy ???????????????????????????

In [ ]:
high_result = train_high_buy_models(model_df)
display(high_result["metrics"])
display(high_result["feature_importance"])

## 13. ????

?? 9 ??????

In [ ]:
clean_old_report_charts()
plot_report_charts(model_df, high_result)
make_wordcloud(merged_df)
for path in sorted(CHART_DIR.glob("*.png")):
    print(path.name)

## 14. ?????????

??????? HTML ???

In [ ]:
generate_dashboard(merged_df, tables, high_result)
print("??????", HTML_DIR / "dashboard.html")

## 15. ???????????

???????????????????????

In [ ]:
print("?????????????????????????????????????????/?????????")
print("?????cat1 ??????????????????????????????????")
print("??????????????????????????????")
print("???????????? VMware?? python ? spark-submit ?????")
print("?????")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))